In [0]:
SILVER_TABLE  = "workspace.new_project.silver_online_retail"
GOLD_COUNTRY  = "workspace.new_project.gold_revenue_by_country"
GOLD_PRODUCTS = "workspace.new_project.gold_top_products"
GOLD_MONTHLY  = "workspace.new_project.gold_monthly_trend"
GOLD_RFM      = "workspace.new_project.gold_customer_rfm"

In [0]:
from pyspark.sql.functions import (
    sum, count, avg, round, col, desc,
    countDistinct, month, year, max, lit,
    datediff, dense_rank
)
from pyspark.sql.window import Window

In [0]:
df = spark.read.table(SILVER_TABLE)

#Gold 1: Revenue by Country

In [0]:
gold_country = df.groupBy("country").agg(
    round(sum("total_amount"), 2).alias("total_revenue"),
    count("invoice_id").alias("total_orders"),
    countDistinct("customer_id").alias("unique_customers"),
    round(avg("total_amount"), 2).alias("avg_order_value")
).orderBy(desc("total_revenue"))

gold_country.write.format("delta").mode("overwrite").saveAsTable(GOLD_COUNTRY)
print(f"{GOLD_COUNTRY}")


workspace.new_project.gold_revenue_by_country


#Gold 2: Top Products

In [0]:
# ── Gold 2: Top Products
product_rev = df.groupBy("stock_code", "description").agg(
    round(sum("total_amount"), 2).alias("total_revenue"),
    sum("quantity").alias("total_quantity")
)
window_spec = Window.orderBy(desc("total_revenue"))
gold_products = product_rev \
    .withColumn("revenue_rank", dense_rank().over(window_spec)) \
    .filter(col("revenue_rank") <= 50)

gold_products.write.format("delta").mode("overwrite").saveAsTable(GOLD_PRODUCTS)
print(f"✅ {GOLD_PRODUCTS}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ workspace.new_project.gold_top_products


#Gold 3: Monthly Trend

In [0]:
# ── Gold 3: Monthly Trend
gold_monthly = df \
    .withColumn("month", month("invoice_date")) \
    .withColumn("year",  year("invoice_date")) \
    .groupBy("year", "month").agg(
        round(sum("total_amount"), 2).alias("revenue"),
        countDistinct("invoice_id").alias("orders"),
        countDistinct("customer_id").alias("customers")
    ).orderBy("year", "month")

gold_monthly.write.format("delta").mode("overwrite").saveAsTable(GOLD_MONTHLY)
print(f"✅ {GOLD_MONTHLY}")

✅ workspace.new_project.gold_monthly_trend


#Gold 4: Customer RFM

In [0]:
max_date = df.agg(max("invoice_date")).collect()[0][0]

gold_rfm = df.groupBy("customer_id").agg(
    datediff(lit(max_date), max("invoice_date")).alias("recency_days"),
    countDistinct("invoice_id").alias("frequency"),
    round(sum("total_amount"), 2).alias("monetary")
)

gold_rfm.write.format("delta").mode("overwrite").saveAsTable(GOLD_RFM)
print(f"✅ {GOLD_RFM}")

print("\n🎯 All Gold tables written successfully!")

✅ workspace.new_project.gold_customer_rfm

🎯 All Gold tables written successfully!


In [0]:
# See all tables in your schema
display(spark.sql("SHOW TABLES IN workspace.new_project"))

database,tableName,isTemporary
new_project,bronze_online_retail,false
new_project,gold_customer_rfm,false
new_project,gold_monthly_trend,false
new_project,gold_revenue_by_country,false
new_project,gold_top_products,false
new_project,online_retail_ecommerce_table,false
new_project,silver_online_retail,false


In [0]:
bronze = spark.read.table("workspace.new_project.bronze_online_retail")
print(f"Bronze rows : {bronze.count():,}")

Bronze rows : 1,067,371


In [0]:
silver = spark.read.table("workspace.new_project.silver_online_retail")
print(f"Silver rows : {silver.count():,}")


Silver rows : 768,862


In [0]:
spark.read.table("workspace.new_project.gold_revenue_by_country").show()


+---------------+-------------+------------+----------------+---------------+
|        country|total_revenue|total_orders|unique_customers|avg_order_value|
+---------------+-------------+------------+----------------+---------------+
| UNITED KINGDOM|1.431078924E7|      689970|            5350|          20.74|
|           EIRE|    616544.54|       15561|               5|          39.62|
|    NETHERLANDS|    553805.51|        5079|              22|         109.04|
|        GERMANY|    423823.42|       16417|             107|          25.82|
|         FRANCE|    348570.34|       13472|              95|          25.87|
|      AUSTRALIA|    169250.26|        1788|              15|          94.66|
|          SPAIN|    108122.23|        3636|              41|          29.74|
|    SWITZERLAND|    100061.94|        3005|              22|           33.3|
|         SWEDEN|     91515.82|        1317|              19|          69.49|
|        DENMARK|     68539.44|         777|              12|   

In [0]:
spark.read.table("workspace.new_project.gold_top_products") \
    .filter("revenue_rank <= 5").show(5, truncate=False)

+----------+----------------------------------+-------------+--------------+------------+
|stock_code|description                       |total_revenue|total_quantity|revenue_rank|
+----------+----------------------------------+-------------+--------------+------------+
|22423     |REGENCY CAKESTAND 3 TIER          |276974.8     |24069         |1           |
|85123A    |WHITE HANGING HEART T-LIGHT HOLDER|245437.41    |91185         |2           |
|23843     |PAPER CRAFT , LITTLE BIRDIE       |168469.6     |80995         |3           |
|M         |Manual                            |143586.94    |8949          |4           |
|85099B    |JUMBO BAG RED RETROSPOT           |134071.23    |74105         |5           |
+----------+----------------------------------+-------------+--------------+------------+



In [0]:
spark.read.table("workspace.new_project.gold_monthly_trend").show(12)


+----+-----+----------+------+---------+
|year|month|   revenue|orders|customers|
+----+-----+----------+------+---------+
|2009|   12| 679214.52|  1512|      955|
|2010|    1| 553701.62|  1011|      720|
|2010|    2|  502429.4|  1104|      772|
|2010|    3| 694418.17|  1524|     1057|
|2010|    4| 589631.03|  1329|      942|
|2010|    5| 594628.38|  1377|      966|
|2010|    6| 632224.16|  1496|     1041|
|2010|    7| 587853.18|  1381|      928|
|2010|    8| 599770.88|  1293|      911|
|2010|    9|  824950.1|  1689|     1145|
|2010|   10|1027025.05|  2133|     1497|
|2010|   11|1157008.14|  2587|     1607|
+----+-----+----------+------+---------+
only showing top 12 rows


In [0]:
spark.read.table("workspace.new_project.gold_customer_rfm").show(5)

+-----------+------------+---------+--------+
|customer_id|recency_days|frequency|monetary|
+-----------+------------+---------+--------+
|      17804|         376|        3|  381.83|
|      17742|         113|       11| 2098.48|
|      16887|          35|        3|  632.67|
|      15380|           8|       11| 3248.96|
|      16210|           1|       35|33750.97|
+-----------+------------+---------+--------+
only showing top 5 rows
